# Tumor Synthesis MVP — MacBook Version
**Strategy:** LDM 학습 대신 Hu et al. copy-paste 합성 (GPU 불필요)

**Reference:** Qixin Hu et al., *Label-free Liver Tumor Segmentation*, CVPR 2023 [proposal ref 10]

## Pipeline
1. **Data inspection** — CT 확인, 통계
2. **Copy-paste synthesis** — 실제 종양을 건강한 CT에 블렌딩
3. **Segmentation training** — synthetic data로 3D U-Net 학습 (MPS/CPU)
4. **Evaluation** — Dice, sensitivity (size-stratified)

## 실행 환경
- MacBook (Apple Silicon MPS 또는 CPU)
- 데이터: `~/tumor-synthesis/data/raw/pancreas/`

## 0. Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.home() / 'tumor-synthesis'
sys.path.insert(0, str(PROJECT_ROOT))

import torch
import numpy as np
import matplotlib.pyplot as plt

# Device: MPS (Apple Silicon) > CPU
if torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')

print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

RAW_DIR    = PROJECT_ROOT / 'data/raw/pancreas'
SYNTH_DIR  = PROJECT_ROOT / 'outputs/samples/pancreas'
CKPT_DIR   = PROJECT_ROOT / 'outputs/checkpoints'
SYNTH_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)
(SYNTH_DIR / 'images').mkdir(exist_ok=True)
(SYNTH_DIR / 'labels').mkdir(exist_ok=True)
(SYNTH_DIR / 'previews').mkdir(exist_ok=True)

In [ ]:
# 데이터 확인
images = sorted((RAW_DIR / 'images').glob('*.nii.gz'))
labels = sorted((RAW_DIR / 'labels').glob('*.nii.gz'))
print(f'Images: {len(images)} | Labels: {len(labels)}')
assert len(images) > 0 and len(labels) > 0

---
## 1. Data Inspection
CT 볼륨 몇 개 시각화해서 데이터 상태 확인.

In [ ]:
import nibabel as nib
from utils.nifti_io import load_nifti, window_ct

# 샘플 3개 시각화
fig, axes = plt.subplots(3, 6, figsize=(18, 9))
for row, (img_p, lbl_p) in enumerate(zip(images[:3], labels[:3])):
    vol, _  = load_nifti(img_p)
    lbl, _  = load_nifti(lbl_p)
    vol_w   = window_ct(vol)
    # tumor exists in label >= 2 for MSD pancreas (1=pancreas, 2=tumor)
    tumor   = (lbl == 2).astype(float)
    tumor_slices = np.where(tumor.sum(axis=(1,2)) > 0)[0]
    if len(tumor_slices) == 0:
        continue
    center = tumor_slices[len(tumor_slices)//2]
    for col, offset in enumerate([-10,-5,0,5,10,15]):
        s = np.clip(center + offset, 0, vol.shape[0]-1)
        axes[row, col].imshow(vol_w[s], cmap='gray', vmin=0, vmax=1)
        if tumor[s].sum() > 0:
            m = np.ma.masked_where(tumor[s]==0, tumor[s])
            axes[row, col].imshow(m, cmap='Reds', alpha=0.5)
        axes[row, col].axis('off')
        axes[row, col].set_title(f'z={s}', fontsize=7)
    print(f'{img_p.name}: shape={vol.shape}, tumor_vox={int(tumor.sum())}')
plt.suptitle('Sample CT volumes (red = tumor)')
plt.tight_layout()
plt.savefig(SYNTH_DIR / 'previews/data_inspection.png', dpi=120)
plt.show()

---
## 2. Copy-Paste Tumor Synthesis
Hu et al. [CVPR 2023] 방식: 실제 종양 영역을 건강한 CT에 Gaussian blending으로 붙여넣기.

**[HU]** 로 표시된 부분이 Hu et al. 기반, **[PROPOSAL]** 은 제안서 기반.

In [ ]:
from scipy.ndimage import gaussian_filter
import random

def extract_tumor_patch(
    volume: np.ndarray,
    mask: np.ndarray,
    margin: int = 8
) -> tuple[np.ndarray, np.ndarray, tuple]:
    """
    [HU] 종양 bounding box + margin 추출.
    Returns (vol_patch, mask_patch, bbox)
    """
    coords = np.argwhere(mask > 0)
    if len(coords) == 0:
        return None, None, None
    z0,y0,x0 = coords.min(axis=0)
    z1,y1,x1 = coords.max(axis=0)
    D,H,W = volume.shape
    z0,y0,x0 = max(0,z0-margin), max(0,y0-margin), max(0,x0-margin)
    z1,y1,x1 = min(D,z1+margin), min(H,y1+margin), min(W,x1+margin)
    return volume[z0:z1,y0:y1,x0:x1], mask[z0:z1,y0:y1,x0:x1], (z0,y0,x0,z1,y1,x1)


def gaussian_blend(
    healthy: np.ndarray,
    tumor_patch: np.ndarray,
    tumor_mask: np.ndarray,
    z0: int, y0: int, x0: int,
    sigma: float = 3.0
) -> tuple[np.ndarray, np.ndarray]:
    """
    [HU] Gaussian feathering으로 경계 자연스럽게 블렌딩.
    Returns (synthetic_volume, binary_mask)
    """
    synth = healthy.copy()
    D,H,W = healthy.shape
    pd,ph,pw = tumor_patch.shape

    # 붙여넣을 영역 클리핑
    z1 = min(z0+pd, D); y1 = min(y0+ph, H); x1 = min(x0+pw, W)
    tp = tumor_patch[:z1-z0, :y1-y0, :x1-x0]
    tm = tumor_mask[:z1-z0, :y1-y0, :x1-x0].astype(float)

    # [HU] Gaussian smoothed mask = blending weight
    blend_w = gaussian_filter(tm, sigma=sigma)
    blend_w = blend_w / (blend_w.max() + 1e-8)

    # intensity alignment: match histogram mean in paste region [PROPOSAL challenge 2]
    healthy_region = healthy[z0:z1, y0:y1, x0:x1]
    if tp[tm > 0.5].size > 0 and healthy_region.size > 0:
        shift = healthy_region.mean() - tp[tm > 0.5].mean()
        tp = tp + shift * 0.5

    synth[z0:z1,y0:y1,x0:x1] = (
        blend_w * tp + (1 - blend_w) * healthy[z0:z1,y0:y1,x0:x1]
    )

    full_mask = np.zeros_like(healthy, dtype=np.uint8)
    full_mask[z0:z1,y0:y1,x0:x1] = (tm > 0.5).astype(np.uint8)
    return synth, full_mask


def find_paste_location(
    healthy: np.ndarray,
    patch_shape: tuple,
    rng: random.Random
) -> tuple[int,int,int]:
    """
    [PROPOSAL] 건강한 CT 볼륨 안에서 랜덤 붙여넣기 위치 선택.
    배경(낮은 HU)은 피하고 복부 영역 안에 배치.
    """
    pd,ph,pw = patch_shape
    D,H,W = healthy.shape
    # 복부 영역 추정: HU > -200인 foreground
    fg = healthy > -200
    pts = np.argwhere(fg)
    if len(pts) == 0:
        pts = np.argwhere(np.ones_like(healthy, dtype=bool))
    # 패치가 들어갈 수 있는 위치만
    valid = pts[(pts[:,0] < D-pd) & (pts[:,1] < H-ph) & (pts[:,2] < W-pw)]
    if len(valid) == 0:
        return 0, 0, 0
    idx = rng.randint(0, len(valid)-1)
    return int(valid[idx,0]), int(valid[idx,1]), int(valid[idx,2])

print('Copy-paste functions defined.')

In [ ]:
from utils.nifti_io import save_nifti
import json

N_SYNTH = 20         # [PROPOSAL] 합성 샘플 수 (MacBook quick run)
rng = random.Random(42)

# 레이블 있는 볼륨 (종양 소스)
labeled_pairs = list(zip(images, labels))
healthy_sources = images[:]

synth_file_list = []
skipped = 0

print(f'Synthesizing {N_SYNTH} tumor volumes (copy-paste)...')

for i in range(N_SYNTH):
    src_img_p, src_lbl_p = rng.choice(labeled_pairs)
    src_vol, affine = load_nifti(src_img_p)
    src_lbl, _ = load_nifti(src_lbl_p)

    tumor_mask = (src_lbl == 2).astype(np.uint8)
    if tumor_mask.sum() < 50:
        skipped += 1
        continue

    t_patch, m_patch, bbox = extract_tumor_patch(src_vol, tumor_mask, margin=6)
    if t_patch is None:
        skipped += 1
        continue

    healthy_p = rng.choice([p for p in healthy_sources if p != src_img_p] or healthy_sources)
    healthy_vol, h_affine = load_nifti(healthy_p)

    z0, y0, x0 = find_paste_location(healthy_vol, t_patch.shape, rng)

    synth_vol, synth_mask = gaussian_blend(healthy_vol, t_patch, m_patch, z0, y0, x0)

    if synth_mask.sum() < 10:
        skipped += 1
        continue

    fname = f'pancreas_synth_{i:05d}'
    img_out = SYNTH_DIR / 'images' / f'{fname}.nii.gz'
    lbl_out = SYNTH_DIR / 'labels' / f'{fname}.nii.gz'
    save_nifti(synth_vol.astype(np.float32), h_affine, img_out)
    save_nifti(synth_mask.astype(np.float32), h_affine, lbl_out)
    synth_file_list.append({'image': str(img_out), 'label': str(lbl_out)})
    print(f'  [{i+1}/{N_SYNTH}] mask_vox={synth_mask.sum()}')

list_path = SYNTH_DIR / 'synth_filelist.json'
with open(list_path, 'w') as f:
    json.dump(synth_file_list, f, indent=2)

print(f'Done: {len(synth_file_list)} synthetic pairs saved, {skipped} skipped.')

In [ ]:
# 합성 결과 시각화 QC
from utils.nifti_io import window_ct

fig, axes = plt.subplots(2, 6, figsize=(18, 6))
for row, entry in enumerate(synth_file_list[:2]):
    vol, _ = load_nifti(entry['image'])
    msk, _ = load_nifti(entry['label'])
    vol_w  = window_ct(vol)
    tumor_slices = np.where(msk.sum(axis=(1,2)) > 0)[0]
    if len(tumor_slices) == 0:
        continue
    center = tumor_slices[len(tumor_slices)//2]
    for col, offset in enumerate([-6,-3,0,3,6,9]):
        s = int(np.clip(center+offset, 0, vol.shape[0]-1))
        axes[row,col].imshow(vol_w[s], cmap='gray', vmin=0, vmax=1)
        if msk[s].sum() > 0:
            m = np.ma.masked_where(msk[s]==0, msk[s])
            axes[row,col].imshow(m, cmap='Reds', alpha=0.5)
        axes[row,col].axis('off')
        axes[row,col].set_title(f'z={s}', fontsize=7)
plt.suptitle('Synthetic tumors — copy-paste [HU] (red = mask)')
plt.tight_layout()
plt.savefig(SYNTH_DIR / 'previews/synth_qc.png', dpi=120)
plt.show()

---
## 3. Segmentation Training
Synthetic (+ optional real) 데이터로 3D U-Net 학습. MPS/CPU 실행.

In [ ]:
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd, Spacingd,
    ScaleIntensityRanged, SpatialPadd, RandSpatialCropd,
    RandFlipd, EnsureTyped, RandCropByPosNegLabeld,
)
from monai.data import Dataset, DataLoader
import torch

PATCH = [48, 48, 48]
BATCH = 2

def make_seg_transforms(is_train: bool, patch_size=PATCH):
    base = [
        LoadImaged(keys=['image','label'], image_only=False),
        EnsureChannelFirstd(keys=['image','label']),
        Orientationd(keys=['image','label'], axcodes='RAS'),
        Spacingd(keys=['image','label'], pixdim=(1.5,1.5,1.5),
                 mode=('bilinear','nearest')),
        ScaleIntensityRanged(keys=['image'],
                             a_min=-150, a_max=250, b_min=0.0, b_max=1.0, clip=True),
        SpatialPadd(keys=['image','label'], spatial_size=patch_size),
        RandCropByPosNegLabeld(
            keys=['image','label'], label_key='label',
            spatial_size=patch_size, pos=3, neg=1,
            num_samples=2, image_key='image', image_threshold=0.0,
        ),
    ]
    if is_train:
        base += [
            RandFlipd(keys=['image','label'], prob=0.5, spatial_axis=0),
            RandFlipd(keys=['image','label'], prob=0.5, spatial_axis=1),
            RandFlipd(keys=['image','label'], prob=0.5, spatial_axis=2),
        ]
    base.append(EnsureTyped(keys=['image','label'],
                             dtype=torch.float32, track_meta=False))
    return Compose(base)

# 학습: synthetic 20개 + real 5개 [PROPOSAL hybrid]
real_pairs = [{'image': str(images[i]), 'label': str(labels[i])}
              for i in range(min(5, len(images)))]
train_files = synth_file_list + real_pairs

# 검증: real 5개
val_files = [{'image': str(images[i]), 'label': str(labels[i])}
             for i in range(5, min(10, len(images)))]

train_ds = Dataset(data=train_files, transform=make_seg_transforms(True))
val_ds   = Dataset(data=val_files,   transform=make_seg_transforms(False))

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                          num_workers=0, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=1, shuffle=False,
                          num_workers=0)

print(f'Train: {len(train_loader)} batches | Val: {len(val_loader)} batches')

In [ ]:
from models.segmentation import TumorSegmenter, get_loss, get_sliding_window_inferer
from monai.metrics import DiceMetric
from monai.transforms import AsDiscrete
import time

seg_model = TumorSegmenter(
    channels=(16, 32, 64, 128, 128),
    strides=(2, 2, 2, 2),
).to(DEVICE)
print(f'Segmenter params: {sum(p.numel() for p in seg_model.parameters()):,}')

loss_fn   = get_loss()
inferer   = get_sliding_window_inferer(PATCH, overlap=0.25)
optimizer = torch.optim.Adam(seg_model.parameters(), lr=1e-4)

MAX_EPOCHS   = 30     # MacBook quick run
VAL_INTERVAL = 5
best_dice    = 0.0
train_losses, val_dices = [], []
seg_ckpt = CKPT_DIR / 'segmenter_macbook_best.pth'

post_pred  = AsDiscrete(argmax=True, to_onehot=2)
post_label = AsDiscrete(to_onehot=2)

In [ ]:
for epoch in range(1, MAX_EPOCHS + 1):
    seg_model.train()
    t0 = time.time()
    ep_loss = 0.0

    for batch in train_loader:
        x = batch['image'].to(DEVICE)
        y = batch['label'].long().to(DEVICE)
        optimizer.zero_grad()
        loss = loss_fn(seg_model(x), y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(seg_model.parameters(), 1.0)
        optimizer.step()
        ep_loss += loss.item()

    ep_loss /= len(train_loader)
    train_losses.append(ep_loss)
    scheduler.step()

    if epoch % VAL_INTERVAL == 0:
        seg_model.eval()
        dice_metric = DiceMetric(include_background=False, reduction='mean')
        with torch.no_grad():
            for batch in val_loader:
                x = batch['image'].to(DEVICE)
                y = batch['label'].long().to(DEVICE)
                out = inferer(x, seg_model)
                dice_metric([post_pred(o) for o in out],
                            [post_label(l) for l in y])
        vdice = dice_metric.aggregate().item()
        val_dices.append((epoch, vdice))
        if vdice > best_dice:
            best_dice = vdice
            torch.save({'model': seg_model.state_dict(), 'epoch': epoch,
                        'best_dice': best_dice}, seg_ckpt)
        print(f'Epoch {epoch:03d}/{MAX_EPOCHS} | '
              f'loss={ep_loss:.4f} | dice={vdice:.4f} | '
              f'{time.time()-t0:.0f}s'
              + (' [saved]' if vdice == best_dice else ''))
    elif epoch % 20 == 0:
        print(f'Epoch {epoch:03d}/{MAX_EPOCHS} | loss={ep_loss:.4f} | {time.time()-t0:.0f}s')

print(f'\nDone. Best Dice: {best_dice:.4f}')

In [ ]:
# 학습 곡선
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(train_losses); axes[0].set_title('Train Loss'); axes[0].grid(True)
if val_dices:
    ep_v, dc_v = zip(*val_dices)
    axes[1].plot(ep_v, dc_v, 'o-'); axes[1].set_title('Val Dice'); axes[1].grid(True)
plt.tight_layout()
plt.savefig(SYNTH_DIR / 'previews/training_curves.png', dpi=120)
plt.show()

---
## 4. Evaluation
Dice, tumor-wise sensitivity (size-stratified), optional Hausdorff. [PROPOSAL metrics]

In [ ]:
from utils.metrics import dice_score, tumor_sensitivity, stratified_sensitivity

ckpt = torch.load(seg_ckpt, map_location=DEVICE)
seg_model.load_state_dict(ckpt['model'])
seg_model.eval()
print(f'Loaded best model (epoch {ckpt["epoch"]}, dice={ckpt["best_dice"]:.4f})')

results = []
with torch.no_grad():
    for batch in val_loader:
        x = batch['image'].to(DEVICE)
        y_np = batch['label'][0,0].cpu().numpy().astype(np.uint8)
        out  = inferer(x, seg_model)
        pred = out[0].argmax(0).cpu().numpy().astype(np.uint8)
        pred_bin = (pred == 1).astype(np.uint8)
        gt_bin   = (y_np == 1).astype(np.uint8)  # label==1 is tumor in synth
        # For real MSD data label 2 = tumor
        gt_bin_real = ((y_np == 1) | (y_np == 2)).astype(np.uint8)
        results.append({
            'dice':        dice_score(pred_bin, gt_bin_real),
            'sensitivity': tumor_sensitivity(pred_bin, gt_bin_real),
            'strat':       stratified_sensitivity(pred_bin, gt_bin_real),
        })

dices = [r['dice'] for r in results]
sens  = [r['sensitivity'] for r in results if not np.isnan(r['sensitivity'])]

print('\n=== EVALUATION RESULTS ===')
print(f'Cases:       {len(results)}')
print(f'Dice:        {np.mean(dices):.4f} ± {np.std(dices):.4f}')
print(f'Sensitivity: {np.mean(sens):.4f}' if sens else 'Sensitivity: N/A')
print('--- Stratified sensitivity ---')
for k in ['small', 'medium', 'large']:
    v = np.nanmean([r['strat'][k] for r in results])
    print(f'  {k:6s}: {v:.4f}')

In [ ]:
# 결과 저장
import json
summary = {
    'method': 'copy_paste_hu_et_al',
    'n_synth': len(synth_file_list),
    'n_val': len(results),
    'dice_mean': float(np.mean(dices)),
    'dice_std':  float(np.std(dices)),
    'sensitivity_mean': float(np.mean(sens)) if sens else None,
    'stratified_sensitivity': {
        k: float(np.nanmean([r['strat'][k] for r in results]))
        for k in ['small','medium','large']
    },
}
out_path = PROJECT_ROOT / 'outputs/eval_macbook.json'
with open(out_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))
print(f'\nSaved to {out_path}')

---
## [PROPOSAL Exp B] Low-annotation ablation
real 데이터 수를 바꾸며 성능 변화 측정.

In [ ]:
# real 데이터 비율 변화: 0(synthetic only), 5개
ablation_results = {}

for n_real in [0, 5]:
    real_subset = [{'image': str(images[i]), 'label': str(labels[i])}
                   for i in range(min(n_real, len(images)))]
    files = synth_file_list + real_subset

    ds  = Dataset(data=files, transform=make_seg_transforms(True))
    ldr = DataLoader(ds, batch_size=BATCH, shuffle=True,
                     num_workers=0, drop_last=True)

    model = TumorSegmenter(channels=(16,32,64,128,128), strides=(2,2,2,2)).to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=1e-4)

    for epoch in range(1, 21):  # 20 epoch quick ablation
        model.train()
        for batch in ldr:
            x = batch['image'].to(DEVICE)
            y = batch['label'].long().to(DEVICE)
            opt.zero_grad()
            loss_fn(model(x), y).backward()
            opt.step()

    model.eval()
    dices_ab = []
    with torch.no_grad():
        for batch in val_loader:
            x = batch['image'].to(DEVICE)
            y_np = batch['label'][0,0].cpu().numpy()
            out  = inferer(x, model)
            pred = out[0].argmax(0).cpu().numpy()
            gt   = ((y_np==1)|(y_np==2)).astype(np.uint8)
            dices_ab.append(dice_score((pred==1).astype(np.uint8), gt))

    ablation_results[f'real_{n_real}'] = float(np.mean(dices_ab))
    print(f'n_real={n_real:3d} | Dice={np.mean(dices_ab):.4f}')

print('\n[PROPOSAL Exp B] Ablation results:')
for k, v in ablation_results.items():
    print(f'  {k}: {v:.4f}')

In [ ]:
# Ablation 그래프
ns = [int(k.split('_')[1]) for k in ablation_results]
ds = list(ablation_results.values())
plt.figure(figsize=(7, 4))
plt.plot(ns, ds, 'o-', color='steelblue', linewidth=2, markersize=8)
plt.xlabel('# Real labeled volumes')
plt.ylabel('Dice score')
plt.title('[Proposal Exp B] Low-annotation: Dice vs. # real volumes')
plt.grid(True)
plt.xticks(ns)
plt.tight_layout()
plt.savefig(SYNTH_DIR / 'previews/ablation_curve.png', dpi=120)
plt.show()